In [6]:
import numpy as np
from LiouvilleLanczos.Quantum_computer.Hamiltonian import Line_Hubbard, BoundaryCondition
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from pauliarray.pauli.weighted_pauli_array import WeightedPauliArray
from pauliarray.pauli.pauli_array import PauliArray
from pauliarray.binary import symplectic 
from pauliarray.binary import bit_operations as bitops
import pauliarray.pauli.pauli_array as pa
from pauliarray.pauli.operator import Operator


In [7]:

def to_pauli(operator):
    mapper = JordanWignerMapper()
    op = mapper.map(operator).simplify(atol=1e-12)

    labels = op.paulis.to_labels()
    weights = op.coeffs
    
    return WeightedPauliArray.from_labels_and_weights(labels=labels, weights=weights), PauliArray.from_labels(labels=labels)

def symplectic_J(n):
    I = np.eye(n, dtype=np.uint8)
    O = np.zeros((n, n), dtype=np.uint8)
    return np.block([
        [O, I],
        [I, O]
    ]).astype(np.uint8)

def zx_to_pauliarray(zx):
    n = zx.shape[1] // 2
    z_strings = zx[:, :n]
    x_strings = zx[:, n:]
    return PauliArray(z_strings, x_strings)

In [8]:
U = 4 
Ham = Line_Hubbard(-1,U/2,U,3,boundary_condition=BoundaryCondition.OPEN)


In [9]:
weight_H, H = to_pauli(Ham)

xz_H = [pauli.xz_strings for pauli in H]            #use xz_strings because bitops does not do symplectic so (z|x)((0 I)(I 0))(Sz|Sx).T: H J S^T so we just flip the zx string to make it symplectic
xz_H = np.array(xz_H)
xz_H = np.vstack(xz_H).astype(int)     #stack xz strings to make one big xz-matrix instead of individual xz strings
print(xz_H)


[[0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 1 1 1 0 0 0]
 [1 0 1 0 0 0 0 1 0 0 0 0]
 [0 0 1 0 1 0 0 0 1 1 1 0]
 [0 0 1 0 1 0 0 0 0 1 0 0]
 [0 1 0 1 0 0 0 1 1 1 0 0]
 [0 1 0 1 0 0 0 0 1 0 0 0]
 [0 0 0 1 0 1 0 0 0 1 1 1]
 [0 0 0 1 0 1 0 0 0 0 1 0]
 [0 0 0 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1]]


In [10]:
ker_H = bitops.kernel(xz_H)                                 ##find the kernel of the hamiltonian to find all paulis that commute with H [S,H] = 0
S = bitops.row_space(ker_H)                                 ##make sure they are linearly independent
iso = symplectic.gram_schmidt_orthogonalization(S)          ##find the generators of the symmetries (isotropic subspace)
conj = symplectic.conjugate_subspace(iso)                   ##find the conjugate subspace to the isotropic subspace [s_i,g_j] = delta_{ij}

print(symplectic.is_orthogonal(iso,conj))                   ##should be false if these anti commute

iso = zx_to_pauliarray(iso)
conj = zx_to_pauliarray(conj)
for i in iso:
    print(i.commute_with(H))                                  ## should all be true since this is the kernel of H they all should commute

print("number of qubits to taper: ",len(iso.to_labels()))


AssertionError: 

In [ ]:
tau = []
sigma = []
for i in range(len(iso.to_labels())):                       ## make sure these are single qubit pauli strings (not the best implementation but just for now)
    if conj[i].num_non_ids == 1:                            ## if number of non identity operators is equal to 1
        tau.append((iso[i].to_labels()))
        sigma.append(conj[i].to_labels())

#------------------------------------------------------------------------------------------------------------------------
# U_i = 1/\sqrt{2}(tau_i + sigma_i) where tau_i is a generator and sigma_i is the single qubit pauli that follows
# [sigma_i,sigma_j] = 0 ; [sigma_i, tau_j] = \delta_{ij}
#------------------------------------------------------------------------------------------------------------------------
U = []
pauli = []
for i in range(len(tau)): 
    s = tau[i]              # just labels
    g = sigma[i]            # just labels
    wp = WeightedPauliArray.from_labels_and_weights([s[0],g[0]],[1/np.sqrt(2),1/np.sqrt(2)]) #manually input the 1/sqrt{2}
    U.append(Operator(wp))          # make it an operator to use clifford_conjugate method from pauliarray
    pauli.append(wp)                # keep track of it as a pauliarray object

In [ ]:
H_tilde = weight_H.copy()
for U_i in U:
    H_tilde = H_tilde.clifford_conjugate(U_i)                   ##clifford conjugate all the symmetry cliffords

print(H_tilde.inspect())

PauliArray
(-3.0000 +0.0000j) IIIIII
(+0.5000 -0.0000j) YXXIII
(-0.5000 +0.0000j) YXXZIZ
(-0.5000 +0.0000j) IYZYII
(-0.5000 +0.0000j) IXZXII
(+0.5000 -0.0000j) ZZXIIZ
(-0.5000 +0.0000j) IIXZXI
(-0.5000 +0.0000j) YZYIII
(-0.5000 +0.0000j) XZXIII
(+1.0000 +0.0000j) ZZZZXI
(+1.0000 +0.0000j) IIZZII
(+1.0000 +0.0000j) ZZIIII
